In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import glob
import numpy as np

In [2]:
parent_folder = "../test"

# retrieve device.imu files from train
imu_test_files = glob.glob(os.path.join(parent_folder, "*", "*", "device_imu.csv"))
print(imu_test_files) 

['../test/2022-02-23-US-LAX-3/XiaomiMi8/device_imu.csv', '../test/2021-09-14-US-MTV-1/GooglePixel5/device_imu.csv', '../test/2021-08-17-US-MTV-1/GooglePixel5/device_imu.csv', '../test/2022-02-23-US-LAX-5/XiaomiMi8/device_imu.csv', '../test/2021-09-20-US-MTV-1/XiaomiMi8/device_imu.csv', '../test/2022-03-14-US-MTV-1/GooglePixel5/device_imu.csv', '../test/2022-04-22-US-OAK-1/GooglePixel5/device_imu.csv', '../test/2022-01-11-US-MTV-1/GooglePixel6Pro/device_imu.csv', '../test/2021-08-24-US-SVL-2/GooglePixel5/device_imu.csv', '../test/2022-04-25-US-OAK-1/GooglePixel5/device_imu.csv', '../test/2022-03-31-US-LAX-3/SamsungGalaxyS20Ultra/device_imu.csv', '../test/2022-02-24-US-LAX-3/XiaomiMi8/device_imu.csv', '../test/2022-01-18-US-SJC-2/GooglePixel5/device_imu.csv', '../test/2022-04-01-US-LAX-3/XiaomiMi8/device_imu.csv', '../test/2022-02-24-US-LAX-5/SamsungGalaxyS20Ultra/device_imu.csv', '../test/2022-02-01-US-SJC-1/XiaomiMi8/device_imu.csv', '../test/2022-03-22-US-MTV-1/SamsungGalaxyS20Ultra/d

In [3]:
imu_list = []

for file in imu_test_files:
    imu_file = pd.read_csv(file)
    parts = file.split(os.sep) # eg ['train', '2021-12-28-US-MTV-1', 'GooglePixel5', 'device_imu.csv']
    imu_file["drive_id"] = parts[-3] 
    imu_file["device"] = parts[-2]
    imu_list.append(imu_file)

# combine all
imu_test_df = pd.concat(imu_list, ignore_index=True)
print(imu_test_df.head())
print()
print(imu_test_df["device"].unique())
print()
print(imu_test_df["drive_id"].unique())

  MessageType  utcTimeMillis  MeasurementX  MeasurementY  MeasurementZ  \
0  UncalAccel  1645655742002     -0.169811      9.894553     -2.316526   
1    UncalMag  1645655742003    -99.040440   -173.611630     -0.707867   
2  UncalAccel  1645655742012     -0.095563      9.671811     -2.009955   
3    UncalMag  1645655742013    -99.040440   -173.611630     -0.707867   
4   UncalGyro  1645655742018     -0.004647     -0.002223     -0.000786   

       BiasX     BiasY    BiasZ             drive_id     device  
0   0.000000    0.0000   0.0000  2022-02-23-US-LAX-3  XiaomiMi8  
1 -83.628395 -180.3668  74.5565  2022-02-23-US-LAX-3  XiaomiMi8  
2   0.000000    0.0000   0.0000  2022-02-23-US-LAX-3  XiaomiMi8  
3 -83.628395 -180.3668  74.5565  2022-02-23-US-LAX-3  XiaomiMi8  
4   0.000000    0.0000   0.0000  2022-02-23-US-LAX-3  XiaomiMi8  

['XiaomiMi8' 'GooglePixel5' 'GooglePixel6Pro' 'SamsungGalaxyS20Ultra'
 'GooglePixel4']

['2022-02-23-US-LAX-3' '2021-09-14-US-MTV-1' '2021-08-17-US-MTV-1'
 '2

In [4]:
imu_testacc_df = imu_test_df[imu_test_df["MessageType"] == "UncalAccel"][["utcTimeMillis", "MeasurementX", "MeasurementY", "MeasurementZ", "drive_id", "device"]]

print(imu_testacc_df.head())

    utcTimeMillis  MeasurementX  MeasurementY  MeasurementZ  \
0   1645655742002     -0.169811      9.894553     -2.316526   
2   1645655742012     -0.095563      9.671811     -2.009955   
5   1645655742022     -0.196157      9.611935     -1.911757   
7   1645655742032     -0.217713      9.770010     -2.201562   
10  1645655742042     -0.153045      9.736478     -2.096178   

               drive_id     device  
0   2022-02-23-US-LAX-3  XiaomiMi8  
2   2022-02-23-US-LAX-3  XiaomiMi8  
5   2022-02-23-US-LAX-3  XiaomiMi8  
7   2022-02-23-US-LAX-3  XiaomiMi8  
10  2022-02-23-US-LAX-3  XiaomiMi8  


In [5]:
imu_testmag_df = pd.DataFrame()

imu_testuncalmag_df = imu_test_df[imu_test_df["MessageType"] == "UncalMag"][["utcTimeMillis", "MeasurementX", "MeasurementY", "MeasurementZ", "BiasX", "BiasY", "BiasZ", "drive_id", "device"]]
imu_testmag_df["utcTimeMillis"] = imu_testuncalmag_df["utcTimeMillis"]
imu_testmag_df["MeasurementX"] = imu_testuncalmag_df["MeasurementX"] + imu_testuncalmag_df["BiasX"]
imu_testmag_df["MeasurementY"] = imu_testuncalmag_df["MeasurementY"] + imu_testuncalmag_df["BiasY"]
imu_testmag_df["MeasurementZ"] = imu_testuncalmag_df["MeasurementZ"] + imu_testuncalmag_df["BiasZ"]
imu_testmag_df["drive_id"] = imu_testuncalmag_df["drive_id"]
imu_testmag_df["device"] = imu_testuncalmag_df["device"]

print(imu_testmag_df.head())

    utcTimeMillis  MeasurementX  MeasurementY  MeasurementZ  \
1   1645655742003   -182.668835    -353.97843     73.848633   
3   1645655742013   -182.668835    -353.97843     73.848633   
6   1645655742023   -184.106645    -354.55757     73.282571   
8   1645655742033   -184.106645    -354.55757     73.282571   
11  1645655742043   -182.623265    -354.81875     71.161257   

               drive_id     device  
1   2022-02-23-US-LAX-3  XiaomiMi8  
3   2022-02-23-US-LAX-3  XiaomiMi8  
6   2022-02-23-US-LAX-3  XiaomiMi8  
8   2022-02-23-US-LAX-3  XiaomiMi8  
11  2022-02-23-US-LAX-3  XiaomiMi8  


In [6]:
imu_testgyro_df = imu_test_df[imu_test_df["MessageType"] == "UncalGyro"][["utcTimeMillis", "MeasurementX", "MeasurementY", "MeasurementZ", "drive_id", "device"]]

print(imu_testgyro_df.head())

    utcTimeMillis  MeasurementX  MeasurementY  MeasurementZ  \
4   1645655742018     -0.004647     -0.002223     -0.000786   
9   1645655742037      0.011067     -0.001690      0.001345   
14  1645655742057      0.006273     -0.001690     -0.000520   
19  1645655742077     -0.003582     -0.002755      0.000013   
24  1645655742097      0.008936     -0.001956      0.000279   

               drive_id     device  
4   2022-02-23-US-LAX-3  XiaomiMi8  
9   2022-02-23-US-LAX-3  XiaomiMi8  
14  2022-02-23-US-LAX-3  XiaomiMi8  
19  2022-02-23-US-LAX-3  XiaomiMi8  
24  2022-02-23-US-LAX-3  XiaomiMi8  


In [7]:
parent_folder = "../test"

# retrieve device.imu files from train
gnss_test_files = glob.glob(os.path.join(parent_folder, "*", "*", "device_gnss.csv"))
print(gnss_test_files) 

['../test/2022-02-23-US-LAX-3/XiaomiMi8/device_gnss.csv', '../test/2021-09-14-US-MTV-1/GooglePixel5/device_gnss.csv', '../test/2021-08-17-US-MTV-1/GooglePixel5/device_gnss.csv', '../test/2022-02-23-US-LAX-5/XiaomiMi8/device_gnss.csv', '../test/2021-09-20-US-MTV-1/XiaomiMi8/device_gnss.csv', '../test/2022-03-14-US-MTV-1/GooglePixel5/device_gnss.csv', '../test/2022-04-22-US-OAK-1/GooglePixel5/device_gnss.csv', '../test/2022-01-11-US-MTV-1/GooglePixel6Pro/device_gnss.csv', '../test/2021-08-24-US-SVL-2/GooglePixel5/device_gnss.csv', '../test/2022-04-25-US-OAK-1/GooglePixel5/device_gnss.csv', '../test/2022-03-31-US-LAX-3/SamsungGalaxyS20Ultra/device_gnss.csv', '../test/2022-02-24-US-LAX-3/XiaomiMi8/device_gnss.csv', '../test/2022-01-18-US-SJC-2/GooglePixel5/device_gnss.csv', '../test/2022-04-01-US-LAX-3/XiaomiMi8/device_gnss.csv', '../test/2022-02-24-US-LAX-5/SamsungGalaxyS20Ultra/device_gnss.csv', '../test/2022-02-01-US-SJC-1/XiaomiMi8/device_gnss.csv', '../test/2022-03-22-US-MTV-1/Samsung

In [8]:
gnss_list = []

for file in gnss_test_files:
    gnss_file = pd.read_csv(file)
    parts = file.split(os.sep) # eg ['train', '2021-12-28-US-MTV-1', 'GooglePixel5', 'device_imu.csv']
    gnss_file["drive_id"] = parts[-3] 
    gnss_file["device"] = parts[-2]
    gnss_list.append(gnss_file)

# combine all
gnss_test_df = pd.concat(gnss_list, ignore_index=True)
print(gnss_test_df.head())
print()
print(gnss_test_df["device"].unique())
print()
print(gnss_test_df["drive_id"].unique())

# shrink large gnss file
gnss_test_df = gnss_test_df[['utcTimeMillis', 'device', 'drive_id', 'Svid', 'Cn0DbHz', 'MultipathIndicator', 'SvElevationDegrees', 'WlsPositionXEcefMeters', 'WlsPositionYEcefMeters', 'WlsPositionZEcefMeters']]

  MessageType  utcTimeMillis    TimeNanos  LeapSecond        FullBiasNanos  \
0         Raw  1645655742000  74816000000         NaN -1329690885184087331   
1         Raw  1645655742000  74816000000         NaN -1329690885184087331   
2         Raw  1645655742000  74816000000         NaN -1329690885184087331   
3         Raw  1645655742000  74816000000         NaN -1329690885184087331   
4         Raw  1645655742000  74816000000         NaN -1329690885184087331   

   BiasNanos  BiasUncertaintyNanos  DriftNanosPerSecond  \
0        0.0              5.985788                  NaN   
1        0.0              5.985788                  NaN   
2        0.0              5.985788                  NaN   
3        0.0              5.985788                  NaN   
4        0.0              5.985788                  NaN   

   DriftUncertaintyNanosPerSecond  HardwareClockDiscontinuityCount  ...  \
0                             NaN                                0  ...   
1                         

In [9]:
def get_cn0dbhz(df_gnss):
    df_gnss = df_gnss.rename(columns={"utcTimeMillis": "UnixTimeMillis"})
    
    cn0dbhz_grp_df = df_gnss.groupby(
        ['drive_id', 'device', 'UnixTimeMillis'],
        as_index=False
    )[[
        'Cn0DbHz'
    ]].mean()
    
    return cn0dbhz_grp_df

cn0dbhz_testmean_df = get_cn0dbhz(gnss_test_df)

print(cn0dbhz_testmean_df)
print()

                  drive_id                 device  UnixTimeMillis    Cn0DbHz
0      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650832999  27.599822
1      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650833999  27.793549
2      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650834999  27.486261
3      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650835999  27.419616
4      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650836999  28.162419
...                    ...                    ...             ...        ...
66092  2022-04-25-US-OAK-2           GooglePixel4   1650927742650  33.194444
66093  2022-04-25-US-OAK-2           GooglePixel4   1650927743642  32.882353
66094  2022-04-25-US-OAK-2           GooglePixel4   1650927744651  32.980952
66095  2022-04-25-US-OAK-2           GooglePixel4   1650927745640  32.850000
66096  2022-04-25-US-OAK-2           GooglePixel4   1650927746632  32.785714

[66097 rows x 4 columns]



In [10]:
def get_satcount(df_gnss):
    df_gnss = df_gnss.rename(columns={"utcTimeMillis": "UnixTimeMillis"})
    
    satcount_grp_df = df_gnss.groupby(['drive_id', 'device', 'UnixTimeMillis'], as_index=False)['Svid'].count()
    
    satcount_grp_df = satcount_grp_df.rename(columns={'Svid': 'SatCount'})
    return satcount_grp_df


def merge_target(merge_df, target_df):
    
    df_merge = merge_df.merge(target_df[['UnixTimeMillis',  'drive_id', 'SatCount', 'device']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'inner')
    
    return df_merge

satcount_testgrp_df = get_satcount(gnss_test_df)
merge_test_df = merge_target(cn0dbhz_testmean_df, satcount_testgrp_df)
print(merge_test_df)
print()

                  drive_id                 device  UnixTimeMillis    Cn0DbHz  \
0      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650832999  27.599822   
1      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650833999  27.793549   
2      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650834999  27.486261   
3      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650835999  27.419616   
4      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650836999  28.162419   
...                    ...                    ...             ...        ...   
66092  2022-04-25-US-OAK-2           GooglePixel4   1650927742650  33.194444   
66093  2022-04-25-US-OAK-2           GooglePixel4   1650927743642  32.882353   
66094  2022-04-25-US-OAK-2           GooglePixel4   1650927744651  32.980952   
66095  2022-04-25-US-OAK-2           GooglePixel4   1650927745640  32.850000   
66096  2022-04-25-US-OAK-2           GooglePixel4   1650927746632  32.785714   

       SatCount  
0            39  
1  

In [11]:
def get_elevationdeg(df_gnss):
    df_gnss = df_gnss.rename(columns={"utcTimeMillis": "UnixTimeMillis"})
    
    elevationdeg_grp_df = df_gnss.groupby(
        ['drive_id', 'device', 'UnixTimeMillis'],
        as_index=False
    )[[
        'SvElevationDegrees'
    ]].mean()
    
    return elevationdeg_grp_df

def merge_target(merge_df, target_df):
    
    df_merge = merge_df.merge(target_df[['UnixTimeMillis',  'drive_id', 'SvElevationDegrees', 'device']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'inner')
    
    return df_merge

elevationdeg_testgrp_df = get_elevationdeg(gnss_test_df)
merge_test_df = merge_target(merge_test_df, elevationdeg_testgrp_df)
print(merge_test_df)
print()

                  drive_id                 device  UnixTimeMillis    Cn0DbHz  \
0      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650832999  27.599822   
1      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650833999  27.793549   
2      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650834999  27.486261   
3      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650835999  27.419616   
4      2021-04-28-US-MTV-2  SamsungGalaxyS20Ultra   1619650836999  28.162419   
...                    ...                    ...             ...        ...   
66092  2022-04-25-US-OAK-2           GooglePixel4   1650927742650  33.194444   
66093  2022-04-25-US-OAK-2           GooglePixel4   1650927743642  32.882353   
66094  2022-04-25-US-OAK-2           GooglePixel4   1650927744651  32.980952   
66095  2022-04-25-US-OAK-2           GooglePixel4   1650927745640  32.850000   
66096  2022-04-25-US-OAK-2           GooglePixel4   1650927746632  32.785714   

       SatCount  SvElevationDegrees  
0

In [12]:
elevationdeg_mean = merge_test_df['SvElevationDegrees'].mean()
print('SvElevationDegrees Mean: ' + str(elevationdeg_mean))

merge_test_df['SvElevationDegrees'] = merge_test_df['SvElevationDegrees'].fillna(elevationdeg_mean)

SvElevationDegrees Mean: 37.98593473680146


In [13]:
def get_tripid(df):
    df['tripId'] = df['drive_id'] + '/' + df['device']
    df.drop(columns=['drive_id', 'device'], inplace=True)
    
    return df

merge_test_df = get_tripid(merge_test_df)
imu_testacc_df = get_tripid(imu_testacc_df)
imu_testgyro_df = get_tripid(imu_testgyro_df)
imu_testmag_df = get_tripid(imu_testmag_df)
gnss_test_df = get_tripid(gnss_test_df)

In [14]:
def get_accmag(merge_df):
    merge_df = merge_df.rename(columns={"utcTimeMillis": "UnixTimeMillis"})
    
    merge_df['AccMag'] = np.sqrt(merge_df['MeasurementX']**2 + merge_df['MeasurementY']**2 + merge_df['MeasurementZ']**2)
    
    return merge_df

def merge_target(merge_df, target_df):
    df_gt = target_df.sort_values('UnixTimeMillis')
    df_acc = merge_df.sort_values('utcTimeMillis')
    df_acc = df_acc.rename(columns={"utcTimeMillis": "UnixTimeMillis"})

    final_df = pd.merge_asof(
        df_gt, 
        df_acc, 
        on='UnixTimeMillis', 
        by='tripId', 
        direction='nearest',
        tolerance=50 
    )
    
    final_df = final_df.sort_values(['tripId', 'UnixTimeMillis']).reset_index(drop=True)
    
    return final_df

acc_testmerge_df = merge_target(imu_testacc_df, merge_test_df)
acc_testmerge_df = get_accmag(acc_testmerge_df)
acc_testmerge_df.drop(columns=['MeasurementX', 'MeasurementY', 'MeasurementZ'], inplace=True)
merge_test_df = acc_testmerge_df
print(merge_test_df)

       UnixTimeMillis    Cn0DbHz  SatCount  SvElevationDegrees  \
0       1619650832999  27.599822        39           38.017116   
1       1619650833999  27.793549        39           38.015823   
2       1619650834999  27.486261        39           37.268639   
3       1619650835999  27.419616        39           37.267143   
4       1619650836999  28.162419        39           38.337918   
...               ...        ...       ...                 ...   
66092   1650927742650  33.194444        18           34.795333   
66093   1650927743642  32.882353        17           35.171023   
66094   1650927744651  32.980952        21           35.621820   
66095   1650927745640  32.850000        20           37.707205   
66096   1650927746632  32.785714        21           35.621205   

                                          tripId    AccMag  
0      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.759161  
1      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.854152  
2      2021-04-28-US-MTV

In [15]:
def get_gyromag(merge_df):
    merge_df = merge_df.rename(columns={"utcTimeMillis": "UnixTimeMillis"})
    
    merge_df['GyroMag'] = np.sqrt(merge_df['MeasurementX']**2 + merge_df['MeasurementY']**2 + merge_df['MeasurementZ']**2)
    
    return merge_df

def merge_target(merge_df, target_df):
    df_gt = target_df.sort_values('UnixTimeMillis')
    df_gyro = merge_df.sort_values('utcTimeMillis')
    df_gyro = df_gyro.rename(columns={"utcTimeMillis": "UnixTimeMillis"})

    final_df = pd.merge_asof(
        df_gt, 
        df_gyro, 
        on='UnixTimeMillis', 
        by='tripId', 
        direction='nearest',
        tolerance=50 
    )
    
    final_df = final_df.sort_values(['tripId', 'UnixTimeMillis']).reset_index(drop=True)
    
    return final_df

gyro_testmerge_df = merge_target(imu_testgyro_df, merge_test_df)
gyro_testmerge_df = get_gyromag(gyro_testmerge_df)
gyro_testmerge_df.drop(columns=['MeasurementX', 'MeasurementY', 'MeasurementZ'], inplace=True)
merge_test_df = gyro_testmerge_df
print(merge_test_df)

       UnixTimeMillis    Cn0DbHz  SatCount  SvElevationDegrees  \
0       1619650832999  27.599822        39           38.017116   
1       1619650833999  27.793549        39           38.015823   
2       1619650834999  27.486261        39           37.268639   
3       1619650835999  27.419616        39           37.267143   
4       1619650836999  28.162419        39           38.337918   
...               ...        ...       ...                 ...   
66092   1650927742650  33.194444        18           34.795333   
66093   1650927743642  32.882353        17           35.171023   
66094   1650927744651  32.980952        21           35.621820   
66095   1650927745640  32.850000        20           37.707205   
66096   1650927746632  32.785714        21           35.621205   

                                          tripId    AccMag   GyroMag  
0      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.759161  0.007877  
1      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.854152  0.013

In [16]:
def get_magmag(merge_df):
    merge_df = merge_df.rename(columns={"utcTimeMillis": "UnixTimeMillis"})
    
    merge_df['MagMag'] = np.sqrt(merge_df['MeasurementX']**2 + merge_df['MeasurementY']**2 + merge_df['MeasurementZ']**2)
    
    return merge_df

def merge_target(merge_df, target_df):
    df_gt = target_df.sort_values('UnixTimeMillis')
    df_mag = merge_df.sort_values('utcTimeMillis')
    df_mag = df_mag.rename(columns={"utcTimeMillis": "UnixTimeMillis"})

    final_df = pd.merge_asof(
        df_gt, 
        df_mag, 
        on='UnixTimeMillis', 
        by='tripId', 
        direction='nearest',
        tolerance=500 
    )
    
    final_df = final_df.sort_values(['tripId', 'UnixTimeMillis']).reset_index(drop=True)
    
    return final_df

mag_testmerge_df = merge_target(imu_testmag_df, merge_test_df)
mag_testmerge_df = get_magmag(mag_testmerge_df)
mag_testmerge_df.drop(columns=['MeasurementX', 'MeasurementY', 'MeasurementZ'], inplace=True)
merge_test_df = mag_testmerge_df
print(merge_test_df)

       UnixTimeMillis    Cn0DbHz  SatCount  SvElevationDegrees  \
0       1619650832999  27.599822        39           38.017116   
1       1619650833999  27.793549        39           38.015823   
2       1619650834999  27.486261        39           37.268639   
3       1619650835999  27.419616        39           37.267143   
4       1619650836999  28.162419        39           38.337918   
...               ...        ...       ...                 ...   
66092   1650927742650  33.194444        18           34.795333   
66093   1650927743642  32.882353        17           35.171023   
66094   1650927744651  32.980952        21           35.621820   
66095   1650927745640  32.850000        20           37.707205   
66096   1650927746632  32.785714        21           35.621205   

                                          tripId    AccMag   GyroMag  \
0      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.759161  0.007877   
1      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.854152  0.0

In [17]:
def get_energy(merge_df):
    merge_df['TotalMotionEnergy'] = np.sqrt(merge_df['AccMag']**2 + merge_df['GyroMag']**2 + merge_df['MagMag']**2)

    return merge_df

merge_test_df = get_energy(merge_test_df)

In [18]:
def get_satcount_delta(merge_df):
    merge_df['SatCountDelta'] = merge_df['SatCount'].diff().fillna(0)

    return merge_df

merge_test_df = get_satcount_delta(merge_test_df)

In [19]:
def get_energy_delta(merge_df):
    merge_df['EnergyDelta'] = merge_df['TotalMotionEnergy'].diff().fillna(0)

    return merge_df

merge_test_df = get_energy_delta(merge_test_df)

In [20]:
def get_cn0dbhz_delta(merge_df):
    merge_df['Cn0DbHzDelta'] = merge_df['Cn0DbHz'].diff().fillna(0)

    return merge_df

merge_test_df = get_cn0dbhz_delta(merge_test_df)

In [21]:
def get_cn0dbhz_mean(merge_df):
    merge_df['Cn0DbHzMean'] =  merge_df['Cn0DbHz'].rolling(window=5).std().fillna(0)

    return merge_df

merge_test_df = get_cn0dbhz_mean(merge_test_df)

In [22]:
def get_energy_std(merge_df):
    merge_df['EnergyStd'] =  merge_df['TotalMotionEnergy'].rolling(window=5).std().fillna(0)

    return merge_df


merge_test_df = get_energy_std(merge_test_df)

In [23]:
def get_wls_val(df_gt, df_gnss):
    gnss_grp_df = df_gnss.groupby(
        ['tripId', 'utcTimeMillis'],
        as_index=False
    )[[
        'WlsPositionXEcefMeters',
        'WlsPositionYEcefMeters',
        'WlsPositionZEcefMeters'
    ]].first()
    
    gnss_renamed_df = gnss_grp_df.rename(columns={"utcTimeMillis": "UnixTimeMillis"})

    df_merge = df_gt.merge(gnss_renamed_df[['UnixTimeMillis', 'WlsPositionXEcefMeters', 'WlsPositionYEcefMeters', 'WlsPositionZEcefMeters', 'tripId']], on = ['UnixTimeMillis', 'tripId'], how = 'left')
    
    return df_merge 

# get wls_val 
merge_test_df = get_wls_val(merge_test_df, gnss_test_df)
print(merge_test_df)

       UnixTimeMillis    Cn0DbHz  SatCount  SvElevationDegrees  \
0       1619650832999  27.599822        39           38.017116   
1       1619650833999  27.793549        39           38.015823   
2       1619650834999  27.486261        39           37.268639   
3       1619650835999  27.419616        39           37.267143   
4       1619650836999  28.162419        39           38.337918   
...               ...        ...       ...                 ...   
66092   1650927742650  33.194444        18           34.795333   
66093   1650927743642  32.882353        17           35.171023   
66094   1650927744651  32.980952        21           35.621820   
66095   1650927745640  32.850000        20           37.707205   
66096   1650927746632  32.785714        21           35.621205   

                                          tripId    AccMag   GyroMag  \
0      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.759161  0.007877   
1      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra  9.854152  0.0

In [24]:
wlsx_mean = merge_test_df['WlsPositionXEcefMeters'].mean()
print('WlsPositionXEcefMeters Mean: ' + str(wlsx_mean))
wlsy_mean = merge_test_df['WlsPositionYEcefMeters'].mean()
print('WlsPositionYEcefMeters Mean: ' + str(wlsy_mean))
wlsz_mean = merge_test_df['WlsPositionZEcefMeters'].mean()
print('WlsPositionZEcefMeters Mean: ' + str(wlsz_mean))

merge_test_df['WlsPositionXEcefMeters'] = merge_test_df['WlsPositionXEcefMeters'].fillna(wlsx_mean)
merge_test_df['WlsPositionYEcefMeters'] = merge_test_df['WlsPositionYEcefMeters'].fillna(wlsy_mean)
merge_test_df['WlsPositionZEcefMeters'] = merge_test_df['WlsPositionZEcefMeters'].fillna(wlsz_mean)

WlsPositionXEcefMeters Mean: -2624539.1356934067
WlsPositionYEcefMeters Mean: -4435674.571659282
WlsPositionZEcefMeters Mean: 3736537.9833721844


In [25]:
print(merge_test_df.isna().sum())

UnixTimeMillis            0
Cn0DbHz                   0
SatCount                  0
SvElevationDegrees        0
tripId                    0
AccMag                    0
GyroMag                   0
MagMag                    0
TotalMotionEnergy         0
SatCountDelta             0
EnergyDelta               0
Cn0DbHzDelta              0
Cn0DbHzMean               0
EnergyStd                 0
WlsPositionXEcefMeters    0
WlsPositionYEcefMeters    0
WlsPositionZEcefMeters    0
dtype: int64


In [26]:
merge_test_df.drop(columns=['AccMag', 'GyroMag', 'MagMag'], inplace=True)

In [27]:
%store merge_test_df

Stored 'merge_test_df' (DataFrame)


In [28]:
test_check = pd.DataFrame()

test_check['tripId'] = merge_test_df['tripId']

%store test_check

Stored 'test_check' (DataFrame)
